# 🪙 Art of Metal — Coin Image Downloader (ZIP Edition)
### Built for Rosalie — No Google Drive needed!

**Steps:**
1. ▶️ Run Cell 1 — install libraries
2. ▶️ Run Cell 2 — upload your Excel file
3. ▶️ Run Cell 3 — fetch all Numista image URLs (takes ~15 min for 833 coins)
4. ▶️ Run Cell 4 — download all images + save ZIP straight to your phone

> ✅ No Google Drive • No copy-pasting • Works on iPhone & Android
> 📦 You'll get one ZIP file with every coin image organized in folders

In [ ]:
# ── CELL 1: Install ────────────────────────────────────────────────────
!pip install openpyxl requests tqdm -q
print('✅ Ready!')

In [ ]:
# ── CELL 2: Upload your Excel file ────────────────────────────────────
from google.colab import files
import pandas as pd
import io

print('📂 Pick your file → Rosalies-Coin-Master-Database.xlsx')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
file_bytes = uploaded[filename]

xl = pd.ExcelFile(io.BytesIO(file_bytes))
print(f'\n✅ Loaded: {filename}')
print(f'📋 Sheets: {xl.sheet_names}')

In [ ]:
# ── CELL 3: Fetch all image URLs ──────────────────────────────────────
import requests, time, re

API_KEY  = 'ioY8OSy8ZHcb7aY5awvPYqC9vMMZ3Zz4G1dSdev0'
HEADERS  = {'User-Agent': 'ArtOfMetal/1.0', 'Numista-API-Key': API_KEY}
DELAY    = 0.7

# Sheets that already have real Numista URLs
BULLION_SHEETS = [
    'Bullion — Gold', 'Bullion — Silver',
    'Bullion — Platinum', 'Bullion — Palladium'
]

# Sheets that need API lookup
CIRC_SHEETS = [
    'Washington Quarter', 'Lincoln Memorial Cent', '50 State Quarters',
    'Morgan Dollar', 'Kennedy Half Dollar', 'Presidential Dollar',
    'Mercury Dime', 'Buffalo Nickel', 'Roosevelt Dime',
    'Peace Dollar', 'Eisenhower Dollar'
]

def safe_name(s):
    return re.sub(r'[^\w\-_. ]', '_', str(s).strip()).replace(' ', '_')[:80]

def numista_lookup(name, year=''):
    q = f'{year} {name}'.strip() if str(year).isdigit() else str(name)
    try:
        r = requests.get(
            'https://api.numista.com/api/v3/coins',
            params={'q': q, 'lang': 'en', 'count': 1},
            headers=HEADERS, timeout=10
        )
        coins = r.json().get('coins', [])
        if not coins:
            return '', ''
        c = coins[0]
        obv = c.get('obverse', {}).get('thumbnail', '') or c.get('obverse', {}).get('picture', '')
        rev = c.get('reverse', {}).get('thumbnail', '') or c.get('reverse', {}).get('picture', '')
        return obv, rev
    except:
        return '', ''

def smithsonian_lookup(name, year=''):
    """Fallback: search Smithsonian Open Access API"""
    try:
        q = f'{year} {name} coin'.strip()
        r = requests.get(
            'https://api.si.edu/openaccess/api/v1.0/search',
            params={'q': q, 'rows': 1, 'media.type': 'Images'},
            timeout=10
        )
        rows = r.json().get('response', {}).get('rows', [])
        if not rows:
            return '', ''
        media = rows[0].get('_imageBucket', {}).get('images', [])
        if media:
            url = media[0].get('content', '')
            return url, ''
        return '', ''
    except:
        return '', ''

# ── Build master image list ────────────────────────────────────────────
all_images = []  # list of (folder, base_name, obv_url, rev_url)

# --- Bullion: already have URLs ---
for sheet in BULLION_SHEETS:
    if sheet not in xl.sheet_names:
        continue
    df = xl.parse(sheet, header=3)
    df = df.dropna(how='all')
    folder = safe_name(sheet)
    count = 0
    for _, row in df.iterrows():
        title = str(row.get('Title', row.get('Numista ID', 'coin')))
        obv   = str(row.get('Obverse Thumb URL', ''))
        rev   = str(row.get('Reverse Thumb URL', ''))
        if obv.startswith('http'):
            all_images.append((folder, safe_name(title), obv, rev))
            count += 1
    print(f'✅ {sheet}: {count} coins queued')

# --- Circulating: fetch from Numista + Smithsonian fallback ---
for sheet in CIRC_SHEETS:
    if sheet not in xl.sheet_names:
        continue
    df = xl.parse(sheet, header=2)
    df = df.dropna(how='all')
    df = df[df.iloc[:, 0].notna()]
    folder = safe_name(sheet)
    found = 0
    print(f'\n🔍 {sheet} ({len(df)} coins)...')

    for _, row in df.iterrows():
        name  = str(row.get('Name', row.get('Title', '')))
        year  = str(row.get('Year', '')).split('.')[0]
        coin_id = str(row.get('ID', name))

        # Check if already has a real URL in sheet
        existing = str(row.get('Obverse Thumb URL', ''))
        if existing.startswith('https://en.numista.com'):
            all_images.append((folder, safe_name(coin_id),
                               existing, str(row.get('Reverse Thumb URL', ''))))
            found += 1
            continue

        # Numista lookup
        obv, rev = numista_lookup(name, year)

        # Smithsonian fallback
        if not obv:
            obv, rev = smithsonian_lookup(name, year)

        status = '✓' if obv else '✗'
        print(f'  {status} {name[:55]}')

        if obv:
            all_images.append((folder, safe_name(coin_id), obv, rev))
            found += 1

        time.sleep(DELAY)

    print(f'  → {found}/{len(df)} images found')

print(f'\n✅ TOTAL: {len(all_images)} coins with image URLs ready to download')

In [ ]:
# ── CELL 4: Download images + ZIP to your phone ───────────────────────
from pathlib import Path
from google.colab import files as colab_files
import zipfile, os

IMG_DIR  = Path('/content/uscoins/coins')
ZIP_PATH = '/content/uscoins/ArtOfMetal_CoinImages.zip'
HEADERS_DL = {'User-Agent': 'ArtOfMetal/1.0'}

IMG_DIR.mkdir(parents=True, exist_ok=True)
dl = skip = fail = 0

def download(url, path):
    if not url or not str(url).startswith('http'):
        return False
    try:
        r = requests.get(url, timeout=30, headers=HEADERS_DL)
        if r.status_code == 200 and len(r.content) > 2000:
            path.write_bytes(r.content)
            return True
    except:
        pass
    return False

print(f'⬇️  Downloading {len(all_images)} coin images...\n')

for folder, base, obv_url, rev_url in all_images:
    d = IMG_DIR / folder
    d.mkdir(parents=True, exist_ok=True)

    ext = obv_url.split('.')[-1][:3] if obv_url.startswith('http') else 'jpg'

    # Obverse
    op = d / f'{base}_obverse.{ext}'
    if op.exists() and op.stat().st_size > 2000:
        skip += 1
    elif download(obv_url, op):
        dl += 1
    else:
        fail += 1

    # Reverse
    if rev_url and rev_url.startswith('http'):
        rp = d / f'{base}_reverse.{ext}'
        if rp.exists() and rp.stat().st_size > 2000:
            skip += 1
        elif download(rev_url, rp):
            dl += 1
        else:
            fail += 1

    time.sleep(0.2)

print(f'\n✅ Downloaded : {dl}')
print(f'⏭  Skipped   : {skip}')
print(f'✗  Failed    : {fail}')

# ── Zip everything ────────────────────────────────────────────────────
print('\n📦 Creating ZIP file...')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in IMG_DIR.rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to(IMG_DIR))

size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f'✅ ZIP ready: {size_mb:.1f} MB')
print('\n📥 Sending to your phone now...')
colab_files.download(ZIP_PATH)
print('✅ Check your Downloads folder!')

---
## 📦 What's in your ZIP
```
ArtOfMetal_CoinImages/
  Bullion_Gold/
    50_Dollars_American_Buffalo_obverse.jpg
    50_Dollars_American_Buffalo_reverse.jpg
    ...
  Bullion_Silver/
    ...
  Washington_Quarter/
    1932_25C_Washington_D_obverse.jpg
    1932_25C_Washington_D_reverse.jpg
    ...
  Morgan_Dollar/
    ...
```
Upload the ZIP to Google Drive, Cloudinary, or wherever you're storing your coin images! 🪙